In [ ]:
from wsidata import open_wsi
import lazyslide as zs

In [ ]:
import os
import pandas as pd

root_dir = r"C:\Users\Admin\Desktop\Christine\slides"
filepaths = []

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.endswith(".mrxs"):
            full_path = os.path.join(root, file)
            filepaths.append(os.path.abspath(full_path))

# create DataFrame
df = pd.DataFrame({"filepath": filepaths})

print(df.head())
print(f"Found {len(df)} slides")

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=42)

labels = np.array([0, 1] * (len(df) // 2))
if len(df) % 2 == 1:
    labels = np.append(labels, rng.integers(0, 2))

rng.shuffle(labels)
df["label"] = labels

In [ ]:
print(df.head())

In [ ]:
recompute = False

if recompute: 
    for file in filepaths:
        wsi = open_wsi(file)
        zs.pp.find_tissues(wsi)
        zs.pp.tile_tissues(wsi, 224, overlap=0.2, background_fraction=0.95, mpp=0.5)
        zs.tl.feature_extraction(wsi, model='conch')
        wsi.write()

In [ ]:
from sklearn.model_selection import train_test_split

# Splitting Data into Training and Validation Sets
train_df, val_df = train_test_split(
    df,
    test_size=0.2,      # 20% of slides for validation
    stratify=df['label'],  # preserve class distribution
    random_state=42
)

In [ ]:
from abmil import ZarrSlideDataset

train_dataset = ZarrSlideDataset(
    df=train_df, 
    filename_col="filepath", 
    label_col="label", 
    feature_key="conch_tiles"
)

val_dataset = ZarrSlideDataset(
    df=val_df,
    filename_col='filepath',
    label_col='label',
    feature_key='conch_tiles'
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [ ]:
from abmil import train_ABMIL

max_tiles = 2000
model = train_ABMIL(train_df=train_df, train_dataset=train_dataset, n_epochs=10, max_tiles=max_tiles)

In [ ]:
from abmil import validate_ABMIL, confusion_matrix_report

all_labels, all_preds = validate_ABMIL(model, val_dataset, max_tiles=max_tiles)
confusion_matrix_report(all_labels, all_preds)

In [ ]:
from abmil import plot_slide_attention_wsiviewer

plot_slide_attention_wsiviewer(model, val_dataset, slide_idx=0, feature_key='conch_tiles')